In [1]:
import os, math, scipy, skbio, statistics
import numpy as np
import pandas as pd
from skbio.diversity import beta_diversity
from skbio.stats.ordination import pcoa
from skbio.stats.composition import clr
import seaborn as sns
import matplotlib as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import fdrcorrection

/tmp/ipykernel_32680/3393769977.py:3: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  abundances = abundances.drop('Unnamed: 0',1)


Unnamed: 0,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Prevotellaceae|g__Prevotella|s__Prevotella_copri_clade_A|t__SGB1626,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcus|s__Ruminococcus_sp_NSJ_71|t__SGB4290,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Lachnospiraceae|g__Coprococcus|s__Coprococcus_eutactus|t__SGB5117,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__GGB3226|s__GGB3226_SGB4260|t__SGB4260,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Ruminococcaceae_unclassified|s__Ruminococcaceae_unclassified_SGB15234|t__SGB15234,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__Phocaeicola|s__Phocaeicola_vulgatus|t__SGB1814,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Rikenellaceae|g__Alistipes|s__Alistipes_putredinis|t__SGB2318,k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales|f__Veillonellaceae|g__Dialister|s__Dialister_invisus|t__SGB5825_group,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii|t__SGB15332_group,k__Bacteria|p__Firmicutes|c__Bacilli|o__Bacilli_unclassified|f__Bacilli_unclassified|g__Bacilli_unclassified|s__Bacilli_unclassified_SGB6571|t__SGB6571,...,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Alcaligenaceae|g__Kerstersia|s__Kerstersia_gyiorum|t__SGB13160,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Clostridiaceae|g__Clostridium|s__Clostridium_mediterraneense|t__SGB21135,k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Eubacteriaceae|g__GGB28645|s__GGB28645_SGB41267|t__SGB41267,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27885|s__GGB27885_SGB40319|t__SGB40319,k__Archaea|p__Candidatus_Thermoplasmatota|c__Thermoplasmata|o__Methanomassiliicoccales|f__Methanomassiliicoccaceae|g__Methanomassiliicoccus|s__Methanomassiliicoccus_luminyensis|t__SGB33442,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae|g__GGB28250|s__GGB28250_SGB40793|t__SGB40793,k__Bacteria|p__Bacteroidetes|c__CFGB4425|o__OFGB4425|f__FGB4425|g__GGB27924|s__GGB27924_SGB40362|t__SGB40362,k__Bacteria|p__Actinobacteria|c__Actinobacteria|o__Propionibacteriales|f__Propionibacteriaceae|g__Propionibacteriaceae_unclassified|s__Propionibacteriaceae_bacterium_NML_150081|t__SGB15922,k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidales_unclassified|g__GGB27883|s__GGB27883_SGB40317|t__SGB40317,k__Bacteria|p__Bacteroidetes|c__CFGB570|o__OFGB570|f__FGB570|g__GGB1201|s__GGB1201_SGB1566|t__SGB1566
MMRS62666492ST-27-0-0,11.54777,6.73408,5.34157,3.17368,2.98767,2.71605,2.62759,2.55274,2.30150,2.12654,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS51737257ST-27-0-0,0.00000,0.67540,0.00041,0.00000,0.00000,4.25769,0.78997,0.00000,1.33619,0.82690,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS38861560ST-27-0-0,0.00561,3.45553,1.53598,0.00000,0.70183,13.45227,2.19867,0.00000,0.57779,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS95674036ST-27-0-0,0.00000,0.00000,0.00000,0.00000,0.00000,18.39640,1.92548,0.95926,0.02935,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MMRS67096717ST-27-0-0,0.00000,0.00133,0.00000,0.00000,0.00000,6.14089,2.10991,1.17805,1.09216,0.03785,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SID72242320_SF07,0.00000,0.00000,1.05172,0.00000,0.88860,5.90205,2.47706,0.07185,0.12916,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SID30920067_SF07,24.37621,0.00000,0.94246,0.00000,0.00000,3.48781,3.01311,0.00000,0.78934,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SID11486372_SF08,0.00000,0.00000,0.00000,0.00000,0.00000,0.01068,3.31002,0.00000,0.31211,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SI

In [3]:
metadata = pd.read_csv('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_metadata.tsv', sep='\t')
metadata

,study_name,sample_id,subject_id,body_site,antibiotics_current_use,study_condition,disease,age,age_category,gender,...,non_westernized,sequencing_platform,PMID,number_reads,number_bases,minimum_read_length,median_read_length,NCBI_accession,curator,DNA_extraction_kit
0,NielsenHB_2014,O2_UC49_0,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,60584637.0,4.099972e+09,30.0,70.0,ERR210493;ERR210492;ERR209641;ERR209640,Paolo_Manghi,NaN
1,NielsenHB_2014,O2_UC49_2,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,56508214.0,3.775477e+09,30.0,70.0,ERR210495;ERR210494;ERR209643;ERR209642,Paolo_Manghi,NaN
2,NielsenHB_2014,O2_UC50_0,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,65105855.0,4.392995e+09,30.0,71.0,ERR210497;ERR210496;ERR209645;ERR209644,Paolo_Manghi,NaN
3,NielsenHB_2014,O2_UC50_2,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,53260679.0,3.596382e+09,30.0,70.0,ERR210500;ERR209648,Paolo_Manghi,NaN
4,NielsenHB_2014,O2_UC51_0,O2_UC51,stool,NaN,control,healthy,32.0,adult,female,...,no,IlluminaHiSeq,24997787.0,57486896.0,3.923766e+09,30.0,72.0,ERR210502;ERR210501;ERR209650;ERR209649,Paolo_Manghi,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12865,BorryM_2020,SAMEA6415059,SAMEA6415059,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12866,BorryM_2020,ERR3761407,ERR3761407,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12867,BorryM_2020,ERR3761411,ERR3761411,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12868,HaganRW_2019,Zape5,Zape5,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
samples_with_profile = sgb_abundances.index.tolist()
samples_with_profile

['MMRS62666492ST-27-0-0',
 'MMRS51737257ST-27-0-0',
 'MMRS38861560ST-27-0-0',
 'MMRS95674036ST-27-0-0',
 'MMRS67096717ST-27-0-0',
 'MMRS84085284ST-27-0-0',
 'MMRS51307502ST-27-0-0',
 'MMRS17963147ST-27-0-0',
 'MMRS39582183ST-27-0-0',
 'MMRS35808664ST-27-0-0',
 'MMRS91297590ST-27-0-0',
 'MMRS95277876ST-27-0-0',
 'MMRS36405092ST-27-0-0',
 'MMRS93621581ST-27-0-0',
 'MMRS97307143ST-27-0-0',
 'MMRS85438660ST-27-0-0',
 'MMRS92546335ST-27-0-0',
 'MMRS33294861ST-27-0-0',
 'MMRS84159866ST-27-0-0',
 'MMRS92727331ST-27-0-0',
 'MMRS95479054ST-27-0-0',
 'MMRS21365932ST-27-0-0',
 'MMRS71238091ST-27-0-0',
 'MMRS34569532ST-27-0-0',
 'MMRS48639115ST-27-0-0',
 'MMRS81135225ST-27-0-0',
 'MMRS25211151ST-27-0-0',
 'MMRS38954404ST-27-0-0',
 'MMRS11664448ST-27-0-0',
 'MMRS39415781ST-27-0-0',
 'MMRS17603756ST-27-0-0',
 'MMRS42570301ST-27-0-0',
 'MMRS11288076ST-27-0-0',
 'MMRS85548821ST-27-0-0',
 'MMRS67690541ST-27-0-0',
 'MMRS29805707ST-27-0-0',
 'MMRS57276462ST-27-0-0',
 'MMRS65862658ST-27-0-0',
 'MMRS616830

In [10]:
filtered_metadata = metadata.loc[metadata['sample_id'].isin(samples_with_profile)]
filtered_metadata

,study_name,sample_id,subject_id,body_site,antibiotics_current_use,study_condition,disease,age,age_category,gender,...,non_westernized,sequencing_platform,PMID,number_reads,number_bases,minimum_read_length,median_read_length,NCBI_accession,curator,DNA_extraction_kit
0,NielsenHB_2014,O2_UC49_0,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,60584637.0,4.099972e+09,30.0,70.0,ERR210493;ERR210492;ERR209641;ERR209640,Paolo_Manghi,NaN
1,NielsenHB_2014,O2_UC49_2,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,56508214.0,3.775477e+09,30.0,70.0,ERR210495;ERR210494;ERR209643;ERR209642,Paolo_Manghi,NaN
2,NielsenHB_2014,O2_UC50_0,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,65105855.0,4.392995e+09,30.0,71.0,ERR210497;ERR210496;ERR209645;ERR209644,Paolo_Manghi,NaN
3,NielsenHB_2014,O2_UC50_2,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,53260679.0,3.596382e+09,30.0,70.0,ERR210500;ERR209648,Paolo_Manghi,NaN
4,NielsenHB_2014,O2_UC51_0,O2_UC51,stool,NaN,control,healthy,32.0,adult,female,...,no,IlluminaHiSeq,24997787.0,57486896.0,3.923766e+09,30.0,72.0,ERR210502;ERR210501;ERR209650;ERR209649,Paolo_Manghi,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12865,BorryM_2020,SAMEA6415059,SAMEA6415059,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12866,BorryM_2020,ERR3761407,ERR3761407,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12867,BorryM_2020,ERR3761411,ERR3761411,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12868,HaganRW_2019,Zape5,Zape5,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
for row in filtered_metadata.index:
    sample_id = filtered_metadata.loc[row]['sample_id']
    study_name = filtered_metadata.loc[row]['study_name']
    if not os.path.exists('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/sample2markers/{}'.format(study_name)):
        os.mkdir('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/sample2markers/{}'.format(study_name))
    org = '/shares/CIBIO-Storage/CM/scratch/data/meta/{}/strainphlan-4beta_vJan21_CHOCOPhlAnSGB_202103/{}/{}.pkl'.format(study_name, sample_id, sample_id)
    dest = '/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/sample2markers/{}/{}.pkl'.format(study_name, sample_id)
    if os.path.exists(org):
        os.symlink(org, dest)
    else:
        print('Missing {}'.format(org))